# Práctica 2: Manejo básico de ROS2

## Objetivo

Que el alumno haga uso de las funciones básicas de ROS2 (publicador y suscriptor) y de sus aplicaciones con turtlesim y rviz2

### Metas

Que el alumno:
- Integre texto en Markdown dentro de Jupyter
- Integre código de Python dentro de Jupyter
- Haga uso de la terminal de Ubuntu y contruya un script de bash
- Genere un repositorio en GitHub y comparta su archivo de práctica

### Contribución al perfil del egresado

La siguiente práctica contribuye en los siguientes puntos al perfil del egresado:

#### Aptitudes y habilidades

- Para diseñar, construir, operar y mantener los sistemas mecatrónicos y sus componentes.
- Para crear, innovar o evaluar las tecnologías relacionadas con la mecatrónica.

#### Actitudes profesionales

- Ser creativo e innovador.
- Tener confianza en su preparación académica.
- Comprometido con su actualización, superación y competencia profesional.

#### Actitudes de tipo social

- Promover el cambio en la mentalidad frente a la competitividad internacional.

## Rúbrica de evaluación

### La evaluación de la práctica contará de los siguientes puntos

| Elemento | Porcentaje |
| ------:| ----------- |
| **Cuestionario previo** | 15% | 
| **Desarrollo** | 35% |
| **Análisis de resultados**  | 35% |
| **Conclusiones** | 15% |

### Se evaluará con los siguientes criterios:

| Elemento | Malo | Regular | Bueno |
| ------:| ------ | --------| ------|
| **Cuestionario previo** | El trabajo no contiene cuestionario previo o todas las preguntas son incorrectas (0%)| Al menos la mitad de las preguntas son correctas (8%) |  Todas las preguntas son correctas (15%) |
| **Desarrollo** | El trabajo no contiene desarrollo o su planteamiento no concuerda con lo deseado (0%) | El desarrollo está mal planteado o no llega a los resultados esperados (10%) | El desarrollo tiene un planteamiento adecuado y llega a los resultados esperados (35%) |
| **Análisis de resultados**  | El trabajo no contiene análisis de resultados o la información no se está interpretando correctamente (0%) | La interpretación de los resultados es parcial o desorganizada (10%) | Realiza un correcto análisis de los resultados de forma organizada   (35%) |
| **Conclusiones** | El trabajo no contiene conclusiones o no hacen referencia al trabajo desarrollado y los objetivos planteados (0%) | La redacción de las conclusiones es desorganizada o confusa (8%) | Las conclusiones del trabajo son claras y hacen referencia al trabajo desarrollado y los objetivos planteados (15%) | 

## Introducción

### ROS2

ROS 2 (Robot Operating System 2) es un conjunto de bibliotecas y herramientas que permiten desarrollar software para robots de forma modular, distribuida y escalable.
En ROS 2, los programas se comunican entre sí mediante mensajes que viajan sobre una red (incluso entre computadoras distintas), usando un sistema de comunicación llamado DDS (Data Distribution Service).

<img src="imagenes/ROS2_logo.webp" alt = "ROS2" height="100" display= "block"/>

#### Nodos

Un nodo es la unidad básica de ejecución en ROS 2.
Cada nodo es un programa que realiza una tarea específica dentro del sistema del robot: leer un sensor, controlar un motor, procesar imágenes, etc.
En Python, los nodos se definen a partir de la clase base rclpy.node.Node.
~~~python
from rclpy.node import Node

class Nodo(Node):
    def __init__(self):
        super().__init__('nombre_nodo')
        self.get_logger().info("Nodo iniciado")
~~~

#### Tópicos

Los tópicos (topics) son canales de comunicación asíncrona y continua entre nodos.
Un nodo publica mensajes en un tópico, y otros nodos se suscriben para recibirlos.
Todos los mensajes que viajan por un tópico tienen un tipo de dato definido, por ejemplo:
- std_msgs/msg/String
- geometry_msgs/msg/Twist

Por ejemplo:

- Un nodo A (sensor) publica temperaturas en el tópico /temperatura.
- Otro nodo B (monitor) se suscribe a /temperatura y muestra los valores.

Para generar un publicador, se crea un publicador definiendo el tipo de mensaje y el tópico por el que se publica. Después se manda a llamar al publicador para mandar un mensaje.
~~~python
  topic_name = "/topic"
  self.publisher = self.create_publisher(String, topic_name, 10)

  msg = String()
  msg.data = "Message"
  self.publisher.publish(msg)
~~~
Para generar un subscriptor:
~~~python 
  topic_name = "/topic"
  self.subscription = self.create_subscription(
    String, topic_name, self.listener_callback, 10)
~~~

#### Servicios

Los servicios (services) son una forma de comunicación sincrónica (de solicitud–respuesta). Funcionan como una llamada a función entre nodos: uno envía una solicitud (request), y el otro responde con un resultado (response).

Los servicios se usan para operaciones puntuales, no continuas (por ejemplo, mover un robot a una posición, tomar una foto, guardar datos, etc.).

Por ejemplo:

- Un nodo A (cliente) pide: “calcula la suma de 2 y 3”.
- Otro nodo B (servidor) responde: “la suma es 5”.

Para crear un servidor, hay que crear el objeto y la función que se llama al recibir una solicitud

~~~python
  self.srv = self.create_service(AddTwoInts, server_name, self.add_two_ints_callback)
def add_two_ints_callback(self, request:AddTwoInts.Request, response:AddTwoInts.Response):
  response.sum = request.a + request.b
  self.get_logger().info("Recibido: a={}, b={}, respuesta={}".format(request.a, request.b, response.sum))
  return response
~~~

Para crear un cliente, hay que crear el objeto del cliente y la solicitud, enviarla, y definir la función que se llamará al recibir el objeto de la respuesta
~~~python
self.client = self.create_client(AddTwoInts, service_name)
# Esperamos al servicio
while not self.client.wait_for_service(timeout_sec=1.0):
  self.get_logger().info('Esperando al servicio {}...'.format(service_name))
request = AddTwoInts.Request()
request.a = self.a
request.b = self.b
# --- Manda solicitud
future = self.client.call_async(request)
# --- Función que se ejecuta al recibir respuesta
future.add_done_callback(self.callback_result)
~~~

#### Parámetros

Los parámetros son valores configurables que un nodo puede leer o modificar durante su ejecución.
Permiten ajustar el comportamiento del nodo sin cambiar el código.

Por ejemplo:

- Un nodo de control de motores puede tener un parámetro max_velocity de 0.1.
- Otro nodo puede cambiarlo a 0.5 sin interrumpir al primer nodo.

Por ejemplo, si en un nodo queremos que el tópico utilizado sea un parámetro
~~~python
self.declare_parameter("topic_param", "/topic")
topic_name = self.get_parameter("topic_param").value
~~~

Una forma gráfica de ver las comunicaciones en ROS2:

<img src="imagenes/Nodes-TopicandService.gif" height = "400" display = "block" alt = "Vosualización de las comunicaciones en ROS2"/>

### URDF (Unified Robot Description Format):

El URDF es un formato de archivo XML que se utiliza para describir la estructura de un robot, comunmente usado en ROS. Permite definir la composición física de un robot, incluyendo sus eslabones y juntas. Además, permite especificar características como el peso, la geometría, y la disposición espacial de los elementos del robot.

Un archivo URDF básico se construye a partir de **eslabones** y **juntas**.

#### Para definir un eslabón, se utiliza la estructura siguiente:
~~~xml
<!--Definición del eslabón-->
<link name = "nombre_del_eslabón">
  <!--
  Elementos visuales del eslabón 
  (modelos o formas geométricas)
  -->
  <visual>
      <!--
      Posición y rotación del elemento visual
      respecto al origen del eslabón
      -->
      <origin xyz = "x y z" rpy = "gamma beta alfa"/> 
        <!--Elementos visuales asociados a la junta-->
        <geometry> 
        <!--Primitiva visual (caja en este caso)-->
        <box size = "x y z"/> 
     </geometry>
        <!--Material usado para el elemento visual-->
        <material name = "nombre_del_material"> 
          <color rgba = "r g b a"/> 
        </material> 
  </visual> 
</link> 
~~~

Los elementos primitivos son formas geométricas básicas que se utilizan para representar las partes de un robot en URDF. Algunos de los más comunes son:

- **Caja (box)**: Un paralelepípedo definido por su ancho, alto y profundidad.
>~~~xml
>  <box size = "x y z"/> 
>~~~
- **Cilindro (cylinder)**: Definido por su radio y longitud.
>~~~xml
>  <cylinder radius = "r" length="l"/> 
>~~~
- **Esfera (sphere)**: Definida por su radio.
>~~~xml
>  <sphere radius = "r"/> 
>~~~
- **Malla (mesh)**: Permite importar geometrías más complejas en formatos como STL, Collada, obj, entre otros.
>~~~xml
>  <mesh filename = "ruta/al/archivo"/> 
>~~~

Se puede agregar cómo se desplegarán estos elementos a través de un **material**: 
- **Color**: Color plano con el que se despliega el elemento, definido por componentes rgb y transparencia (alfa).
>~~~xml
> <material name = "Nombre"> 
>   <color rgba = "r g b a"/> 
> </material> 
>~~~
- **Textura**: Imagen que se despliega sobre el modelo.
>~~~xml
> <material name = "Nombre"> 
>   <texture filename = "ruta/al/archivo"/> 
> </material> 
>~~~

#### Para una junta: 
~~~xml
<!--Definición de la junta-->
<joint name = "nombre_de_la_junta" type = "tipo_de_junta"> 
  <!--
  Eslabones padre e hijo
  - Padre: Eslabón respecto al cual se mide la rotació o traslación
  - Hijo: Eslabón que se desplaza o rota
  -->
  <parent link = "eslabon_padre_(fijo)"/> 
  <child link = "eslabon_hijo_(movible)"/> 
  <origin xyz = "x y z" rpy = "gamma beta alfa"/> 
  <!--Eje de rotación  (1: se mueve. 0: no se mueve)-->
  <axis xyz = "x y z"/> 
  <!--Límites 
  - effort: par máximo
  - velocity: velocidad absoluta máxima
  - lower-upper: posición mínima y máxima-->
  <limit effort="10.0" lower="-3.14" upper="3.14" velocity="3.14"/>
</joint>
~~~
En el formato URDF, existen varios tipos de juntas que pueden definir el movimiento relativo entre las partes de un robot:

- **Junta fija**: No permite ningún movimiento entre los elementos conectados.
>~~~xml
> <joint name = "nombre_de_la_junta" type = "fixed"> 
>~~~
- **Junta rotacional**: Permite un movimiento de rotación alrededor de un eje.
>~~~xml
> <joint name = "nombre_de_la_junta" type = "revolute"> 
>~~~
- **Junta prismática**: Permite un movimiento lineal a lo largo de un eje.
>~~~xml
> <joint name = "nombre_de_la_junta" type = "prismatic"> 
>~~~
- **Junta continua**: Es una variación de la junta rotacional que permite una rotación sin límites.
>~~~xml
> <joint name = "nombre_de_la_junta" type = "continuous"> 
>~~~

### RVIZ


RViz (ROS Visualization) es una herramienta gráfica dentro del ecosistema de ROS que permite visualizar una amplia gama de información generada por robots. Esta herramienta es esencial para depurar, entender y visualizar los datos que se generan en tiempo real durante la simulación o la operación de un robot en un entorno físico.

<img src="imagenes/rviz.png" alt = "RViz" height="300" display= "block"/>


RViz soporta muchos tipos de visualizaciones, en este caso veremos:

- **Modelos 3D del robot (URDF)**: Puedes cargar y visualizar la descripción del robot usando su archivo URDF (Unified Robot Description Format). Esto te permite ver un modelo 3D del robot y observar cómo se mueve cada parte del robot (es decir, los eslabones y las juntas). También puedes ver cómo se actualiza en tiempo real cuando el robot cambia de posición.

- **Transformaciones (TF)**: ROS utiliza un sistema de coordenadas llamado TF (transformación de marcos) para definir la relación espacial entre diferentes partes del robot y el entorno. En RViz, puedes visualizar estos sistemas de referencia y las transformaciones entre ellos, lo que es útil para entender cómo se mueven y orientan los diferentes componentes del robot.
Esto incluye visualizar el origen de coordenadas del robot y sus partes móviles.

## Cuestionario previo

### Responder de forma breve las siguientes preguntas:

- ¿Qué es un ejecutor (executor) en ROS2?
> Es el componente que se encarga de ejecutar los nodos y sus callbacks, gestionando el flujo de mensajes entre publicadores, suscriptores y servicios.
- ¿Qué es la ejecución por hilos en python?
> Permite ejecutar varias tareas al mismo tiempo dentro de un mismo programa, usando distintos hilos (threads) de procesamiento.
- ¿Qué ventajas tiene ejecutar un programa en un hilo secundario?
> Permite no bloquear el programa principal, haciendo que otras tareas sigan funcionando mientras una operación larga o en espera se ejecuta aparte.
- ¿Qué pasaría si se utiliza un comando que se queda en periodo de espera (como rclpy.spin()) dentro de un entorno de celdas como jupyter?
> El cuaderno se quedaría bloqueado, ya que rclpy.spin() entra en un bucle infinito y detiene la ejecución de nuevas celdas hasta que el proceso termine o se interrumpa manualmente.

*En caso de integrar imagenes, colocarlas en la carpeta "imagenes"*

## Desarrollo

### 1. Tópicos en ROS2

En este primera parte, se usarán las funciones básicas de ROS2 dentro de jupyter

Para ello, en celdas separadas se realizarán las siguientes acciones:

- Inicializar ros2 e importar dependencias (nodos, mensajes, ejecución por hilos y ejecutor de ros2)
~~~python
import rclpy
from rclpy.node import Node
from std_msgs.msg import String
import threading
from rclpy.executors import MultiThreadedExecutor
~~~
- Crear una clase que herede de nodo, que implemente un publicador y sus funciones
~~~python
  class Publisher(Node):
    def __init__(self):
      # Constructor de inicio
    def timer_callback(self):
      # Función de timer para publicar periódicamente cada segundo
~~~
- Crear una clase que herede de nodo, que implemente un subscriptor y sus funciones
~~~python
  class Subscriber(Node):
    def __init__(self):
      # Constructor
    def listener_callback(self, msg):
      # Función que se llama al recibir un mensaje
~~~
- Verificar si ROS2 está inicializado. Si no, inicializarlo. Utilizar la función rclpy.ok(), que nos devuelve el estado de ROS2
~~~python
  if not rclpy.ok():
    rclpy.init(args=None)
~~~
- Instanciar un ejecutor de ROS2 que se encargue de varios nodos de forma simultánea, y ejecutarlo en segundo plano para no bloquear la terminal
~~~ python
  executor = MultiThreadedExecutor()
  thread = threading.Thread(target=executor.spin, daemon=True)
  thread.start()
~~~
- Instanciar los nodos publicador y suscriptor y agregarlos al ejecutor
~~~python
  pub_node = SimplePublisher()
  sub_node = SimpleSubscriber()
  executor.add_node(sub_node)
  executor.add_node(pub_node)
~~~
- Destruir los nodos para que dejen de ejecutarse
~~~python
  pub_node.destroy_node()
  sub_node.destroy_node()
~~~
- Que detenga el proceso de ROS2 en caso de que siga activo (para evitar procesos en segundo plano)
~~~python
  if rclpy.ok():
    rclpy.shutdown()
~~~

Como probatorio: 
- Las celdas con código en esta sección del archivo
- Dejar los mensajes de salida que se imprimirán al agregar los nodos al ejecutor

In [21]:
import rclpy
from rclpy.node import Node
from std_msgs.msg import String
import threading
from rclpy.executors import MultiThreadedExecutor

print("Dependencias importadas.")

Dependencias importadas.


In [22]:
# 2. Crear una clase que herede de nodo (Publicador)
class PublicadorSimple(Node):
    def __init__(self):
        # Constructor de inicio
        super().__init__('publicador_simple')
        # Creamos el publicador. El tópico se llamará 'charla'
        self.publicador_ = self.create_publisher(String, 'charla', 10)
        self.periodo_timer = 1.0  # 1 segundo
        self.temporizador = self.create_timer(self.periodo_timer, self.funcion_timer)
        self.contador = 0
        self.get_logger().info('Nodo publicador inicializado.')

    def funcion_timer(self):
        # Función de timer para publicar periódicamente cada segundo
        mensaje = String()
        mensaje.data = f'Hola ROS2 desde Jupyter: {self.contador}'
        self.publicador_.publish(mensaje)
        self.get_logger().info(f'Publicando: "{mensaje.data}"')
        self.contador += 1

In [23]:
# 3. Crear una clase que herede de nodo (Subscriptor)
class SubscriptorSimple(Node):
    def __init__(self):
        # Constructor
        super().__init__('subscriptor_simple')
        self.subscripcion_ = self.create_subscription(
            String,
            'charla',  # Debe coincidir con el tópico del publicador
            self.funcion_escucha,
            10)
        self.subscripcion_  # (evitar warning de variable no usada)
        self.get_logger().info('Nodo subscriptor inicializado.')

    def funcion_escucha(self, mensaje):
        # Función que se llama al recibir un mensaje
        self.get_logger().info(f'He recibido: "{mensaje.data}"')

In [8]:
# 4. Verificar si ROS2 está inicializado. Si no, inicializarlo.
if not rclpy.ok():
    rclpy.init(args=None)
    print("Contexto de ROS2 inicializado.")
else:
    print("ROS2 ya estaba inicializado.")

Contexto de ROS2 inicializado.


In [9]:
# 5. Instanciar un ejecutor multi-hilo y ejecutarlo en segundo plano
ejecutor = MultiThreadedExecutor()
hilo = threading.Thread(target=ejecutor.spin, daemon=True)
hilo.start()
print("Ejecutor iniciado en un hilo (daemon) separado.")

Ejecutor iniciado en un hilo (daemon) separado.


In [10]:
# 6. Instanciar los nodos publicador y suscriptor y agregarlos al ejecutor
nodo_publicador = PublicadorSimple()
nodo_subscriptor = SubscriptorSimple()

ejecutor.add_node(nodo_subscriptor)
ejecutor.add_node(nodo_publicador)

print("Nodos agregados al ejecutor. La comunicación debería comenzar.")

Nodos agregados al ejecutor. La comunicación debería comenzar.


[INFO] [1762536773.712108250] [publicador_simple]: Nodo publicador inicializado.
[INFO] [1762536773.720989927] [subscriptor_simple]: Nodo subscriptor inicializado.
[INFO] [1762536774.677944002] [subscriptor_simple]: He recibido: "Hola ROS2 desde Jupyter: 0"
[INFO] [1762536774.683867388] [publicador_simple]: Publicando: "Hola ROS2 desde Jupyter: 0"
[INFO] [1762536775.687510294] [publicador_simple]: Publicando: "Hola ROS2 desde Jupyter: 1"
[INFO] [1762536775.693298867] [subscriptor_simple]: He recibido: "Hola ROS2 desde Jupyter: 1"
[INFO] [1762536776.689610890] [subscriptor_simple]: He recibido: "Hola ROS2 desde Jupyter: 2"
[INFO] [1762536776.696609728] [publicador_simple]: Publicando: "Hola ROS2 desde Jupyter: 2"
[INFO] [1762536777.983112902] [publicador_simple]: Publicando: "Hola ROS2 desde Jupyter: 3"
[INFO] [1762536777.987544451] [subscriptor_simple]: He recibido: "Hola ROS2 desde Jupyter: 3"
[INFO] [1762536778.731042104] [subscriptor_simple]: He recibido: "Hola ROS2 desde Jupyter: 4

In [11]:
# 7. Destruir los nodos para que dejen de ejecutarse
print("Deteniendo los nodos...")
nodo_publicador.destroy_node()
nodo_subscriptor.destroy_node()
print("Nodos destruidos.")

Deteniendo los nodos...
Nodos destruidos.


In [12]:
# 8. Que detenga el proceso de ROS2 en caso de que siga activo
if rclpy.ok():
    print("Apagando ROS2 (shutdown)...")
    rclpy.shutdown()
    print("ROS2 apagado.")
else:
    print("ROS2 ya estaba apagado.")

Apagando ROS2 (shutdown)...
ROS2 apagado.


### 2. Servicios en ROS2

En este segunda parte, se usarán los servicios de ROS2 dentro de jupyter

Para ello, en celdas separadas se realizarán las siguientes acciones:

- Inicializar ros2 e importar dependencias (nodos, mensajes, ejecución por hilos y ejecutor de ros2)
~~~python
import rclpy
from rclpy.node import Node
from std_msgs.msg import String
import threading
from rclpy.executors import MultiThreadedExecutor
~~~
- Crear una clase que herede de nodo, que implemente un servidor y sus funciones
~~~python
  class Server(Node):
    def __init__(self):
      # Constructor
    def listener_callback(self, msg):
      # Función que se llama al recibir un mensaje
~~~
- Crear una clase que herede de nodo, que implemente un cliente y sus funciones
~~~python
  class Client(Node):
    def __init__(self):
      # Constructor de inicio
    def timer_callback(self):
      # Función de timer para enviar una solicitud periódicamente cada segundo
    def callback_result(self, future:Future):
      # Función que se llama al recibir una respuesta a la solicitud enviada
~~~
- Verificar si ROS2 está inicializado. Si no, inicializarlo. Utilizar la función rclpy.ok(), que nos devuelve el estado de ROS2
~~~python
  if not rclpy.ok():
    rclpy.init(args=None)
~~~
- Instanciar un ejecutor de ROS2 que se encargue de varios nodos de forma simultánea, y ejecutarlo en segundo plano para no bloquear la terminal
~~~ python
  executor = MultiThreadedExecutor()
  thread = threading.Thread(target=executor.spin, daemon=True)
  thread.start()
~~~
- Instanciar los nodos servidor y cliente, y agregarlos al ejecutor
~~~python
  server_node = Server()
  client_node = Client()
  executor.add_node(server_node)
  executor.add_node(client_node)
~~~
- Destruir los nodos para que dejen de ejecutarse
~~~python
  server_node.destroy_node()
  client_node.destroy_node()
~~~
- Que detenga el proceso de ROS2 en caso de que siga activo (para evitar procesos en segundo plano)
~~~python
  if rclpy.ok():
    rclpy.shutdown()
~~~

Como probatorio: 
- Las celdas con código en esta sección del archivo
- Dejar los mensajes de salida que se imprimirán al agregar los nodos al ejecutor

In [13]:
# 1. Importar dependencias
import rclpy
from rclpy.node import Node
import threading
from rclpy.executors import MultiThreadedExecutor

# Importamos el tipo de servicio de ejemplo (Sumar Dos Enteros)
from example_interfaces.srv import AddTwoInts
# Importamos 'Future' para la gestión de la respuesta del cliente
from rclpy.task import Future

print("Dependencias importadas (incluyendo servicio AddTwoInts).")

Dependencias importadas (incluyendo servicio AddTwoInts).


In [14]:
# 2. Crear una clase que herede de nodo (Servidor)
class Servidor(Node):
    def __init__(self):
        # Constructor
        super().__init__('servidor_sumas')
        # Se crea el servicio 'sumar_enteros'
        self.srv = self.create_service(
            AddTwoInts, 
            'sumar_enteros', 
            self.funcion_callback_servicio)
        self.get_logger().info('Nodo Servidor inicializado y listo para recibir peticiones.')

    def funcion_callback_servicio(self, peticion, respuesta):
        # Función que se llama al recibir una petición (request)
        # 'peticion' es el objeto AddTwoInts.Request
        # 'respuesta' es el objeto AddTwoInts.Response
        
        respuesta.sum = peticion.a + peticion.b
        
        self.get_logger().info(f'Petición recibida: a={peticion.a}, b={peticion.b}')
        self.get_logger().info(f'Enviando respuesta: {respuesta.sum}')
        
        return respuesta

In [15]:
# 3. Crear una clase que herede de nodo (Cliente)
class Cliente(Node):
    def __init__(self):
        # Constructor de inicio
        super().__init__('cliente_sumas')
        # Se crea el cliente. Debe coincidir el tipo y nombre del servicio
        self.cliente = self.create_client(AddTwoInts, 'sumar_enteros')
        
        # Esperamos a que el servicio esté disponible
        while not self.cliente.wait_for_service(timeout_sec=1.0):
            self.get_logger().info('Servicio no disponible, esperando...')
        
        self.get_logger().info('Nodo Cliente inicializado. El servicio está listo.')
        self.contador_peticion = 1
        
        # Creamos el timer para enviar peticiones periódicamente
        self.temporizador = self.create_timer(1.0, self.funcion_timer)

    def funcion_timer(self):
        # Función de timer para enviar una solicitud periódicamente
        if not self.cliente.service_is_ready():
            self.get_logger().warn('El servicio se ha desconectado. Reintentando...')
            return

        # Creamos el objeto de petición (Request)
        peticion = AddTwoInts.Request()
        peticion.a = self.contador_peticion
        peticion.b = self.contador_peticion * 2
        
        self.get_logger().info(f'Enviando petición: {peticion.a} + {peticion.b}')
        
        # Llamamos al servicio de forma asíncrona
        self.futuro = self.cliente.call_async(peticion)
        # Asignamos la función que se llamará cuando llegue la respuesta
        self.futuro.add_done_callback(self.callback_resultado)
        
        self.contador_peticion += 1

    def callback_resultado(self, futuro: Future):
        # Función que se llama al recibir una respuesta
        try:
            respuesta = futuro.result()
            self.get_logger().info(f'Resultado recibido: {respuesta.sum}')
        except Exception as e:
            self.get_logger().error(f'Llamada al servicio falló: {e}')

In [16]:
# 4. Verificar si ROS2 está inicializado. Si no, inicializarlo.
if not rclpy.ok():
    rclpy.init(args=None)
    print("Contexto de ROS2 inicializado.")
else:
    print("ROS2 ya estaba inicializado.")

Contexto de ROS2 inicializado.


In [17]:
# 5. Instanciar un ejecutor multi-hilo y ejecutarlo en segundo plano
ejecutor = MultiThreadedExecutor()
hilo = threading.Thread(target=ejecutor.spin, daemon=True)
hilo.start()
print("Ejecutor iniciado en un hilo (daemon) separado.")

Ejecutor iniciado en un hilo (daemon) separado.


In [18]:
# 6. Instanciar los nodos servidor y cliente y agregarlos al ejecutor

# Primero instanciamos el servidor
nodo_servidor = Servidor()
ejecutor.add_node(nodo_servidor)

# Esperamos un momento y luego el cliente
# (El cliente tiene un 'wait_for_service' interno, así que esto es seguro)
nodo_cliente = Cliente()
ejecutor.add_node(nodo_cliente)

print("Nodos Servidor y Cliente agregados al ejecutor.")

Nodos Servidor y Cliente agregados al ejecutor.


[INFO] [1762537213.540000380] [servidor_sumas]: Nodo Servidor inicializado y listo para recibir peticiones.
[INFO] [1762537213.554026197] [cliente_sumas]: Nodo Cliente inicializado. El servicio está listo.
[INFO] [1762537214.565500289] [cliente_sumas]: Enviando petición: 1 + 2
[INFO] [1762537214.575481260] [servidor_sumas]: Petición recibida: a=1, b=2
[INFO] [1762537214.580497869] [servidor_sumas]: Enviando respuesta: 3
[INFO] [1762537214.587175304] [cliente_sumas]: Resultado recibido: 3
[INFO] [1762537215.568519049] [cliente_sumas]: Enviando petición: 2 + 4
[INFO] [1762537215.574221194] [servidor_sumas]: Petición recibida: a=2, b=4
[INFO] [1762537215.577953949] [servidor_sumas]: Enviando respuesta: 6
[INFO] [1762537215.583187415] [cliente_sumas]: Resultado recibido: 6
[INFO] [1762537216.575306844] [cliente_sumas]: Enviando petición: 3 + 6
[INFO] [1762537216.587138661] [servidor_sumas]: Petición recibida: a=3, b=6
[INFO] [1762537216.592973138] [servidor_sumas]: Enviando respuesta: 9
[I

In [19]:
# 7. Destruir los nodos para que dejen de ejecutarse
print("Deteniendo los nodos...")
nodo_servidor.destroy_node()
nodo_cliente.destroy_node()
print("Nodos destruidos.")

Deteniendo los nodos...
Nodos destruidos.


In [20]:
# 8. Que detenga el proceso de ROS2 en caso de que siga activo
if rclpy.ok():
    print("Apagando ROS2 (shutdown)...")
    rclpy.shutdown()
    print("ROS2 apagado.")
else:
    print("ROS2 ya estaba apagado.")

Apagando ROS2 (shutdown)...
ROS2 apagado.


### 2. URDF y RViz

En esta sección, se creará un archivo URDF de un robot tipo RRR con la siguiente disposición:
<div align="center">
<img src="imagenes/modelo_robot.png" alt = "Robot RRR" width="300" height="300"/>
</div>

** Se pueden utilizar formas geométricas sencillas
** Mantener las dimensiones menores a 0.5 \[m]. 

#### Despliegue de un modelo URDF con configuraciones predeterminadas en RViz
Ahora, se tomará el archivo URDF realizado y se desplegará en RViz utilizando algunas configuraciones predeterminadas. 
Para desplegar el archivo en RViz, se deben instalar algunas librerías:

Para descargar un paquete que contiene un launch para desplegar al robot
> $ sudo apt-get install ros-humble-urdf-tutorial

Instalar la librería que permite a ROS manipular los URDF (xacro)
> $ sudo apt install ros-humble-xacro

Hay que verificar las juntas del archivo. RViz requiere que las juntas tengan límites definidos, agregando el parámetro
``` xml 
<limit effort="XX" velocity="XX" lower="XX" upper="XX" />
```
Siendo "XX" los valores deseados para cada parámetro.

Con estos cambios, se puede correr el despliegue del modelo con un archivo *.launch del paquete urdf-tutorial 

> $ ros2 launch urdf_tutorial display.launch.py model:=/home/robousr/<ruta_del_modelo>/<nombre_del_modelo>.urdf



Como entregable, agregar una imagen del modelo desplegado en RViz y una celda con el código xml del modelo urdf

<div align="center">
<img src="rrp.png" alt = "Primer despliegue en RViz" width="600" height="600" display= "block"/>
</div>

~~~xml
<?xml version="1.0"?>
<robot name="custom_robot_r_r_p_stacked_arms">

  <material name="blue">
    <color rgba="0.2 0.5 0.8 1.0"/>
  </material>
  <material name="silver">
    <color rgba="0.7 0.7 0.7 1.0"/>
  </material>
  <material name="dark_grey">
    <color rgba="0.3 0.3 0.3 1.0"/>
  </material>

  <link name="world"/>
  
  <joint name="world_to_base" type="fixed">
    <parent link="world"/>
    <child link="base_link"/>
  </joint>

  <link name="base_link">
    <visual>
      <geometry>
        <cylinder radius="0.15" length="0.05"/>
      </geometry>
      <origin xyz="0 0 0.025" rpy="0 0 0"/>
      <material name="blue"/>
    </visual>
    
    <visual>
      <geometry>
        <cylinder radius="0.04" length="0.3"/>
      </geometry>
      <origin xyz="0 0 0.2" rpy="0 0 0"/> 
      <material name="silver"/>
    </visual>
  </link>

  <joint name="joint_1_rot_V" type="revolute">
    <parent link="base_link"/>
    <child link="link_1"/>
    <origin xyz="0 0 0.35" rpy="0 0 0"/>
    <axis xyz="0 0 1"/>
    <limit lower="-3.14" upper="3.14" effort="10" velocity="1"/>
  </joint>

  <link name="link_1">
    <visual>
      <geometry>
        <box size="0.2 0.04 0.04"/>
      </geometry>
      <origin xyz="0.1 0 0" rpy="0 0 0"/>
      <material name="blue"/>
    </visual>
  </link>

  <joint name="joint_2_rot_R" type="revolute">
    <parent link="link_1"/>
    <child link="link_2"/>
    <origin xyz="0.2 0 0.02" rpy="0 0 0"/> 
    <axis xyz="0 0 1"/>
    <limit lower="-3.14" upper="3.14" effort="10" velocity="1"/>
  </joint>

  <link name="link_2">
    <visual>
      <geometry>
        <box size="0.2 0.04 0.04"/>
      </geometry>
      <origin xyz="0.05 0 0.02" rpy="0 0 0"/>
      <material name="blue"/>
    </visual>
  </link>

  <joint name="joint_3_prism_O" type="prismatic">
    <parent link="link_2"/>
    <child link="end_effector_link"/>
    <origin xyz="0.15 0 0.15" rpy="0 0 0"/>
    <axis xyz="0 0 1"/>
    <limit lower="-0.15" upper="0.0" effort="5" velocity="0.5"/>
  </joint>

  <link name="end_effector_link">
    <visual>
      <geometry>
        <cylinder radius="0.01" length="0.25"/>
      </geometry>
      <origin xyz="0 0 -0.05" rpy="0 0 0"/>
      <material name="dark_grey"/>
    </visual>
  </link>

</robot>
~~~

## Análisis de resultados

¿Cuáles son las ventajas o desventajas de usar el protocolo de comunicación publicador/suscriptor ó cliente/servicio?
> El protocolo publicador/suscriptor en ROS2 ofrece la ventaja de permitir una comunicación continua y asíncrona entre nodos, su principal desventaja es que no existe una confirmación directa de que el mensaje haya sido recibido.
> el protocolo cliente/servicio establece una comunicación que garantiza que el nodo que realiza la petición reciba un resultado, su principal desventaja es que no es eficiente cuando se necesita transmitir datos de manera continua.

¿Cuál es la convención de ángulos que utilizan los archivos URDF para los ángulos de las juntas respecto al eslabón padre? (Intrínsecos/extrínsecos y el orden)
> Utilizan ángulos extrínsecos en el orden roll–pitch–yaw (X–Y–Z) para definir la orientación de una junta respecto al eslabón padre.

¿Qué utilidad tiene describir un robot en un archivo URDF?
> Permite representar la estructura física y geométrica del robot, facilitando su visualización en Rviz2 y la simulación en Gazebo.




## Conclusiones

La práctica permitió comprender de manera integral el funcionamiento del sistema ROS2 y su importancia como herramienta para el desarrollo de aplicaciones robóticas modulares y distribuidas. A través del uso de publicadores, suscriptores, servicios y parámetros, se comprobó cómo los nodos pueden comunicarse de forma eficiente y cómo la arquitectura de ROS2 facilita la interacción entre distintos componentes de un robot.

El trabajo con Turtlesim y Rviz2 permitió visualizar los resultados de los mensajes y las transformaciones espaciales, reforzando la comprensión del flujo de datos entre nodos y la utilidad de la representación visual en entornos simulados.



## Bibliografía 

Hacer referencia a la información implementada en formato ieee, en caso de haberse utilizado